# 02 Clean and merge
Stage 2: put the three sources on one monthly time axis. Input: `data/`. Output: `data/clean_monthly.csv`.

In [1]:
import sys
sys.path.insert(0, "code")
import numpy as np
import pandas as pd
import helpers as h

## 1. Swiss fuel indices (UST), monthly, December 2020 = 100

In [2]:
ust = h.load_ust_fuel(h.DATA / "su-d-05_02_67.xlsx")
print(ust.shape, ust.index.min().date(), ust.index.max().date())
print(ust.isna().sum())   # petrol and diesel are empty before December 2005

(517, 4) 1982-12-01 2025-12-01
diesel         276
petrol         276
fuels_total      0
cpi_total        0
dtype: int64


## 2. Brent and USD/CHF (FRED), daily

In [3]:
def read_fred(path, col):
    df = pd.read_csv(path, parse_dates=[0], index_col=0)
    s = pd.to_numeric(df[col], errors="coerce")      # FRED writes missing days as '.'
    print(col, "rows:", len(s), "missing:", int(s.isna().sum()), s.index.min().date(), s.index.max().date())
    return s

brent = read_fred(h.DATA / "brent_usd_daily.csv", "DCOILBRENTEU")
chf_per_usd = read_fred(h.DATA / "chf_per_usd_daily.csv", "DEXSZUS")

DCOILBRENTEU rows: 10305 missing: 1173 1987-05-20 2026-11-17
DEXSZUS rows: 14575 missing: 566 1971-01-04 2026-11-13


## 3. Monthly means and merge
Missing days are skipped in the monthly mean; a month with no quotation at all would stay missing (checked below). Brent in francs = Brent in USD times CHF per USD.

In [4]:
m = pd.DataFrame({
    "brent_usd": h.to_monthly_mean(brent),
    "chf_per_usd": h.to_monthly_mean(chf_per_usd),
})
m["brent_chf"] = m["brent_usd"] * m["chf_per_usd"]
df = ust.join(m, how="inner")
print(df.shape, df.index.min().date(), df.index.max().date())
print(df.isna().sum())

(517, 7) 1982-12-01 2025-12-01
diesel         276
petrol         276
fuels_total      0
cpi_total        0
brent_usd       53
chf_per_usd      0
brent_chf       53
dtype: int64


## 4. Logs and monthly log changes
Price indices and Brent enter in logs, so that coefficients read as elasticities and changes as percentages.

In [5]:
cols = ["diesel", "petrol", "fuels_total", "cpi_total", "brent_usd", "chf_per_usd", "brent_chf"]
logs = np.log(df[cols]).add_prefix("log_")
dlogs = logs.diff().add_prefix("d")
out = pd.concat([df, logs, dlogs], axis=1)
out.to_csv(h.DATA / "clean_monthly.csv")
print(out.loc["2005-12-01":, ["log_diesel", "log_petrol", "log_brent_chf"]].dropna().shape)
out.tail(3).round(3)

(241, 3)


,diesel,petrol,fuels_total,cpi_total,brent_usd,chf_per_usd,brent_chf,log_diesel,log_petrol,log_fuels_total,...,log_brent_usd,log_chf_per_usd,log_brent_chf,dlog_diesel,dlog_petrol,dlog_fuels_total,dlog_cpi_total,dlog_brent_usd,dlog_chf_per_usd,dlog_brent_chf
2025-10-01,118.620,119.373,119.151,107.177,64.543,0.797,51.470,4.776,4.782,4.780,...,4.167,-0.226,3.941,-0.004,-0.006,-0.005,-0.003,-0.052,0.002,-0.050
2025-11-01,120.087,119.316,119.578,106.952,63.797,0.804,51.272,4.788,4.782,4.784,...,4.156,-0.219,3.937,0.012,-0.000,0.004,-0.002,-0.012,0.008,-0.004
2025-12-01,121.765,119.791,120.436,106.938,62.544,0.797,49.842,4.802,4.786,4.791,...,4.136,-0.227,3.909,0.014,0.004,0.007,-0.000,-0.020,-0.008,-0.028
